# Landsat Tiles - Forest East Of Coimbra

Notebook Landsat-only per ispezionare tile Earth Engine sull'area forestale a est di Coimbra.

Usa solo:

- `LANDSAT/LC08/C02/T1_L2`
- `LANDSAT/LC09/C02/T1_L2`

e solo scene `PROCESSING_LEVEL == "L2SP"`, perche' la temperatura superficiale usa `ST_B10`.

Output del notebook:

- tile URL Landsat RGB;
- tile URL Landsat falso colore NIR;
- tile URL LST per acquisizione singola;
- tile URL LST mediana stagionale;
- mappa Folium con layer controllabili.

In [1]:
from __future__ import annotations

import datetime as dt
import os
from pprint import pprint

import certifi

# Use the CA bundle from certifi to avoid macOS/Python SSL certificate issues during EE auth.
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

import ee
import folium
import pandas as pd

# Replace with your Earth Engine / Google Cloud project ID.
EARTH_ENGINE_PROJECT = "eo4c-498809"
AUTH_MODE = "localhost"  # local Mac/VS Code notebook. Alternatives: "notebook", "gcloud".

# Approximate forest domain east of Coimbra: Serra da Lousa / Miranda do Corvo / Gois.
# Coordinates are lon/lat in EPSG:4326: west, south, east, north.
AOI_BBOX = [-8.35, 40.00, -7.75, 40.38]
MAP_CENTER = [40.18, -8.05]

START_DATE = "2025-06-01"
END_DATE = "2025-09-30"
MAX_SCENE_CLOUD_COVER = 70

MIN_CLOUD_DISTANCE_KM = 1.0
MAX_ST_UNCERTAINTY_K = 2.0
MASK_WATER = True

## 1. Authenticate Earth Engine

La prima volta si aprira' il login. Nessuna credenziale viene salvata nel notebook.

In [2]:
def initialize_earth_engine(project: str, auth_mode: str = "localhost") -> None:
    """Authenticate and initialize Earth Engine before creating EE objects."""
    has_project = bool(project and project != "replace-with-project-id")
    try:
        if has_project:
            ee.Initialize(project=project)
        else:
            ee.Initialize()
    except Exception:
        ee.Authenticate(auth_mode=auth_mode)
        if has_project:
            ee.Initialize(project=project)
        else:
            ee.Initialize()


initialize_earth_engine(EARTH_ENGINE_PROJECT, AUTH_MODE)

# Create Earth Engine geometries only after ee.Initialize().
aoi = ee.Geometry.Rectangle(AOI_BBOX)
aoi_area_km2 = aoi.area(maxError=1).divide(1_000_000).getInfo()

{
    "status": "Earth Engine initialized",
    "project": EARTH_ENGINE_PROJECT,
    "aoi_area_km2": round(aoi_area_km2, 2),
}


Successfully saved authorization token.


/Users/nicolocaron/Documents/GitHub/EO4C/.venv-eo/lib/python3.13/site-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(
/Users/nicolocaron/Documents/GitHub/EO4C/.venv-eo/lib/python3.13/site-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


{'status': 'Earth Engine initialized',
 'project': 'eo4c-498809',
 'aoi_area_km2': 2153.5}

## 2. Landsat preprocessing helpers

Queste funzioni sono intenzionalmente Landsat-only. La maschera LST usa `QA_PIXEL`, `QA_RADSAT`, `ST_CDIST` e `ST_QA`.

In [3]:
def qa_bit_clear(qa: ee.Image, bit: int) -> ee.Image:
    return qa.bitwiseAnd(1 << bit).eq(0)


def landsat_quality_mask(image: ee.Image) -> ee.Image:
    """Apply Landsat L2SP QA mask for optical and thermal preview layers."""
    qa = image.select("QA_PIXEL")
    radsat = image.select("QA_RADSAT")
    st_uncertainty_k = image.select("ST_QA").multiply(0.01).rename("ST_uncertainty_K")
    cloud_distance_km = image.select("ST_CDIST").multiply(0.01).rename("cloud_distance_km")

    mask = (
        qa_bit_clear(qa, 0)  # fill
        .And(qa_bit_clear(qa, 1))  # dilated cloud
        .And(qa_bit_clear(qa, 2))  # high-confidence cirrus
        .And(qa_bit_clear(qa, 3))  # cloud
        .And(qa_bit_clear(qa, 4))  # cloud shadow
        .And(qa_bit_clear(qa, 5))  # snow / ice
        .And(radsat.eq(0))
        .And(cloud_distance_km.gte(MIN_CLOUD_DISTANCE_KM))
        .And(st_uncertainty_k.lte(MAX_ST_UNCERTAINTY_K))
    )
    if MASK_WATER:
        mask = mask.And(qa_bit_clear(qa, 7))

    return image.addBands([st_uncertainty_k, cloud_distance_km], overwrite=True).updateMask(mask)


def add_scaled_reflectance(image: ee.Image) -> ee.Image:
    """Add scaled optical surface reflectance bands for map display."""
    sr = image.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5"]).multiply(0.0000275).add(-0.2)
    return image.addBands(sr, overwrite=True)


def add_lst_celsius(image: ee.Image) -> ee.Image:
    """Convert Landsat ST_B10 DN to LST in degrees Celsius."""
    lst_c = image.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).rename("LST_C")
    return image.addBands(lst_c, overwrite=True)


def preprocess_landsat(image: ee.Image) -> ee.Image:
    image = landsat_quality_mask(image)
    image = add_scaled_reflectance(image)
    image = add_lst_celsius(image)
    return image.copyProperties(
        image,
        [
            "system:time_start",
            "SPACECRAFT_ID",
            "LANDSAT_PRODUCT_ID",
            "PROCESSING_LEVEL",
            "CLOUD_COVER",
            "WRS_PATH",
            "WRS_ROW",
        ],
    )

## 3. Load Landsat 8 and Landsat 9 L2SP scenes

Landsat 8 e Landsat 9 vengono filtrati separatamente, preprocessati con la stessa funzione e poi uniti.

In [4]:
def load_landsat_l2sp(collection_id: str) -> ee.ImageCollection:
    return (
        ee.ImageCollection(collection_id)
        .filterBounds(aoi)
        .filterDate(START_DATE, END_DATE)
        .filter(ee.Filter.lte("CLOUD_COVER", MAX_SCENE_CLOUD_COVER))
        .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))
    )


landsat8_raw = load_landsat_l2sp("LANDSAT/LC08/C02/T1_L2")
landsat9_raw = load_landsat_l2sp("LANDSAT/LC09/C02/T1_L2")

landsat8 = landsat8_raw.map(preprocess_landsat)
landsat9 = landsat9_raw.map(preprocess_landsat)
landsat = landsat8.merge(landsat9).sort("system:time_start")

counts = {
    "landsat8_l2sp": landsat8_raw.size().getInfo(),
    "landsat9_l2sp": landsat9_raw.size().getInfo(),
    "merged_l2sp": landsat.size().getInfo(),
}
counts

{'landsat8_l2sp': 15, 'landsat9_l2sp': 14, 'merged_l2sp': 29}

In [5]:
def image_summary_table(collection: ee.ImageCollection, limit: int = 12) -> pd.DataFrame:
    """Return a small metadata table for selected acquisitions."""
    features = collection.limit(limit).map(
        lambda image: ee.Feature(
            None,
            {
                "date": ee.Date(image.get("system:time_start")).format("YYYY-MM-dd"),
                "satellite": image.get("SPACECRAFT_ID"),
                "product_id": image.get("LANDSAT_PRODUCT_ID"),
                "cloud_cover": image.get("CLOUD_COVER"),
                "wrs_path": image.get("WRS_PATH"),
                "wrs_row": image.get("WRS_ROW"),
            },
        )
    )
    rows = [feature["properties"] for feature in features.getInfo()["features"]]
    return pd.DataFrame(rows)


image_summary_table(landsat, limit=20)

,cloud_cover,date,product_id,satellite,wrs_path,wrs_row
0,49.22,2025-06-05,LC08_L2SP_204032_20250605_20250607_02_T1,LANDSAT_8,204,32
1,12.86,2025-06-06,LC09_L2SP_203032_20250606_20250609_02_T1,LANDSAT_9,203,32
2,63.17,2025-06-13,LC09_L2SP_204032_20250613_20250614_02_T1,LANDSAT_9,204,32
3,3.93,2025-06-14,LC08_L2SP_203032_20250614_20250626_02_T1,LANDSAT_8,203,32
4,43.52,2025-06-21,LC08_L2SP_204032_20250621_20250627_02_T1,LANDSAT_8,204,32
5,8.78,2025-06-22,LC09_L2SP_203032_20250622_20250623_02_T1,LANDSAT_9,203,32
6,38.56,2025-06-29,LC09_L2SP_204032_20250629_20250630_02_T1,LANDSAT_9,204,32
7,0.55,2025-06-30,LC08_L2SP_203032_20250630_20250711_02_T1,LANDSAT_8,203,32
8,1.05,2025-07-07,LC08_L2SP_204032_20250707_20250715_02_T1,LANDSAT_8,204,32
9,0.01,2025-07-08,LC09_L2SP_203032_20250708_20250710_02_T1,LANDSAT_9,203,32


## 4. Build preview images

`single_scene` e' la scena meno nuvolosa del periodo. `seasonal_median` e' la mediana del periodo estivo scelto.

In [6]:
single_scene = ee.Image(landsat.sort("CLOUD_COVER").first()).clip(aoi)
seasonal_median = landsat.median().clip(aoi)

valid_observations = landsat.select("LST_C").count().rename("valid_observations").clip(aoi)

single_scene_info = single_scene.toDictionary(
    ["system:time_start", "SPACECRAFT_ID", "LANDSAT_PRODUCT_ID", "CLOUD_COVER", "WRS_PATH", "WRS_ROW"]
).getInfo()
single_scene_info["date"] = dt.datetime.utcfromtimestamp(single_scene_info["system:time_start"] / 1000).date().isoformat()
single_scene_info

/Users/nicolocaron/Documents/GitHub/EO4C/.venv-eo/lib/python3.13/site-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(
/var/folders/q5/g4rsymgj6137bp73lx06r3g40000gn/T/ipykernel_853/2977071461.py:9: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  single_scene_info["date"] = dt.datetime.utcfromtimestamp(single_scene_info["system:time_start"] / 1000).date().isoformat()


{'CLOUD_COVER': 0,
 'LANDSAT_PRODUCT_ID': 'LC08_L2SP_203032_20250716_20250726_02_T1',
 'SPACECRAFT_ID': 'LANDSAT_8',
 'WRS_PATH': 203,
 'WRS_ROW': 32,
 'system:time_start': 1752664066365,
 'date': '2025-07-16'}

## 5. Tile URL examples from Landsat

Questi URL sono tile XYZ Earth Engine per prototipo. Sono utili nel notebook e per sviluppo, ma per pubblicazione stabile useremo COG + tile service.

In [7]:
VIS_RGB = {"bands": ["SR_B4", "SR_B3", "SR_B2"], "min": 0.02, "max": 0.35, "gamma": 1.15}
VIS_FALSE_COLOR = {"bands": ["SR_B5", "SR_B4", "SR_B3"], "min": 0.02, "max": 0.45, "gamma": 1.15}
VIS_LST = {
    "bands": ["LST_C"],
    "min": 15,
    "max": 45,
    "palette": ["#313695", "#4575b4", "#74add1", "#ffffbf", "#fdae61", "#d73027", "#a50026"],
}
VIS_VALID = {"bands": ["valid_observations"], "min": 1, "max": 10, "palette": ["#ffffcc", "#41b6c4", "#0c2c84"]}


def tile_url(image: ee.Image, vis: dict) -> str:
    return image.getMapId(vis)["tile_fetcher"].url_format


tile_examples = [
    {
        "layer": "Landsat seasonal RGB median",
        "source": "LC08/LC09 C02 T1 L2 L2SP",
        "tile_url": tile_url(seasonal_median, VIS_RGB),
        "visualization": VIS_RGB,
    },
    {
        "layer": "Landsat seasonal NIR false color median",
        "source": "LC08/LC09 C02 T1 L2 L2SP",
        "tile_url": tile_url(seasonal_median, VIS_FALSE_COLOR),
        "visualization": VIS_FALSE_COLOR,
    },
    {
        "layer": "Landsat single-scene LST C",
        "source": "ST_B10 converted to Celsius",
        "tile_url": tile_url(single_scene, VIS_LST),
        "visualization": VIS_LST,
    },
    {
        "layer": "Landsat seasonal median LST C",
        "source": "ST_B10 converted to Celsius",
        "tile_url": tile_url(seasonal_median, VIS_LST),
        "visualization": VIS_LST,
    },
    {
        "layer": "Valid Landsat LST observations",
        "source": "Count of valid masked LST pixels",
        "tile_url": tile_url(valid_observations, VIS_VALID),
        "visualization": VIS_VALID,
    },
]

pd.DataFrame(tile_examples)[["layer", "source", "tile_url"]]

,layer,source,tile_url
0,Landsat seasonal RGB median,LC08/LC09 C02 T1 L2 L2SP,https://earthengine.googleapis.com/v1/projects...
1,Landsat seasonal NIR false color median,LC08/LC09 C02 T1 L2 L2SP,https://earthengine.googleapis.com/v1/projects...
2,Landsat single-scene LST C,ST_B10 converted to Celsius,https://earthengine.googleapis.com/v1/projects...
3,Landsat seasonal median LST C,ST_B10 converted to Celsius,https://earthengine.googleapis.com/v1/projects...
4,Valid Landsat LST observations,Count of valid masked LST pixels,https://earthengine.googleapis.com/v1/projects...


## 6. Interactive map

Apri il controllo layer in alto a destra. La LST scientifica non e' clippata; `min` e `max` sono solo parametri di visualizzazione della tile.

In [8]:
def add_ee_tile(map_object: folium.Map, image: ee.Image, vis: dict, name: str, shown: bool = False, opacity: float = 1.0) -> None:
    map_id = image.getMapId(vis)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        name=name,
        attr="Google Earth Engine / USGS Landsat Collection 2 Level-2",
        overlay=True,
        control=True,
        show=shown,
        opacity=opacity,
    ).add_to(map_object)


m = folium.Map(location=MAP_CENTER, zoom_start=11, tiles="OpenStreetMap", control_scale=True)

add_ee_tile(m, seasonal_median, VIS_RGB, "Landsat RGB seasonal median", shown=True)
add_ee_tile(m, seasonal_median, VIS_FALSE_COLOR, "Landsat NIR false color seasonal median", shown=False)
add_ee_tile(m, single_scene, VIS_LST, "Landsat LST C - least cloudy scene", shown=False, opacity=0.75)
add_ee_tile(m, seasonal_median, VIS_LST, "Landsat LST C - seasonal median", shown=True, opacity=0.65)
add_ee_tile(m, valid_observations, VIS_VALID, "Valid LST observation count", shown=False, opacity=0.75)

folium.GeoJson(
    data=aoi.getInfo(),
    name="AOI east of Coimbra",
    style_function=lambda _: {"color": "#111111", "weight": 2, "fillOpacity": 0},
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

/Users/nicolocaron/Documents/GitHub/EO4C/.venv-eo/lib/python3.13/site-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


## 7. What to use next

Quando il bounding box e il periodo sono corretti, il prossimo passo e' sostituire `AOI_BBOX` con il GeoJSON reale dell'area forestale e far girare la pipeline production in `project/` per esportare COG e CSV.